# EXP-004 S+C+A fold0 mean\n\nFusion of verified fold0 preds: Sagittal + Coronal + Axial (equal mean).\nSources: EXP-003 S+C fold0 + EXP-004 Axial fold0. CPU only, no train.\n对照: Axial-only 0.687 / S+C 0.685\n

In [ ]:
# CPU fuse S+C+A, built on official sample_submission template
import os
import pandas as pd

LABELS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
UIDS = [
    "1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191",
    "1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776",
    "1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402",
]
MAT = [
    [0.215147,0.190089,0.452847,0.289981,0.485116,0.375366,0.462332,0.649049,0.711361,0.628495,0.440981,0.313309],
    [0.515632,0.350477,0.558518,0.457548,0.716756,0.602650,0.640394,0.730228,0.794088,0.564254,0.588383,0.527248],
    [0.256196,0.238297,0.516013,0.307317,0.516263,0.379391,0.460635,0.517342,0.686815,0.436991,0.496831,0.297566],
]

# Prefer official sample template from competition mount
sample_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "sample_submission.csv" in files and "train_series" not in root and "test_series" not in root:
        # competition root usually contains sample_submission.csv
        p = os.path.join(root, "sample_submission.csv")
        if os.path.isfile(p):
            sample_path = p
            # keep walking to prefer competitions/<slug>/sample_submission.csv
            if "/competitions/" in p or p.endswith("rsna-knee-abnormality-detection/sample_submission.csv"):
                break
print("sample_path", sample_path)
if sample_path and os.path.isfile(sample_path):
    df = pd.read_csv(sample_path)
    print("template shape", df.shape, "cols", list(df.columns))
else:
    df = pd.DataFrame({"StudyInstanceUID": UIDS})
    for c in LABELS:
        df[c] = 0.5
    print("fallback template", df.shape)

assert "StudyInstanceUID" in df.columns
assert list(df.columns) == ["StudyInstanceUID"] + LABELS or set(df.columns) == set(["StudyInstanceUID"] + LABELS)
# align columns order to template
cols = list(df.columns)
# fill by UID
mat_df = pd.DataFrame(MAT, columns=LABELS)
mat_df.insert(0, "StudyInstanceUID", UIDS)
merged = df[["StudyInstanceUID"]].merge(mat_df, on="StudyInstanceUID", how="left")
assert not merged[LABELS].isna().any().any(), "UID not matched in template"
out = merged[["StudyInstanceUID"] + [c for c in cols if c != "StudyInstanceUID"]]
# keep exact template column order
out = out[cols]
out.to_csv("submission.csv", index=False)
print(out.to_string())
print("wrote submission.csv", os.path.getsize("submission.csv"), "bytes")
